# Custom Word2Vec Training Notebook

This notebook demonstrates how to train a custom Word2Vec model from scratch using text data. We will cover data loading, preprocessing, model training, embedding exploration, and model evaluation.

## 1. Import Required Libraries

Import necessary libraries including gensim, nltk, pandas, numpy, and matplotlib for Word2Vec training and visualization.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Gensim for Word2Vec
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

# NLTK for text processing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import WordNetLemmatizer

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

print("All libraries imported successfully!")

## 2. Load and Preprocess Text Data

Load text data from files or sources and perform initial preprocessing such as removing special characters, converting to lowercase, and handling missing values.

In [ ]:
# Load text data from file
def load_text_data(filepath):
    """Load text data from a file."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
        print(f"Text data loaded successfully from {filepath}")
        print(f"Total characters: {len(text)}")
        return text
    except FileNotFoundError:
        print(f"File not found: {filepath}")
        return None

# For this example, we'll use the story.txt file from the data directory
text_data = load_text_data('assignments/0607word2vec/data/story.txt')

# Display first 500 characters
if text_data:
    print("\nFirst 500 characters of the text:")
    print(text_data[:500])

## 3. Tokenize and Clean Text

Tokenize sentences into words, remove stopwords, apply lemmatization or stemming, and filter out low-frequency words.

In [ ]:
def preprocess_text(text):
    """
    Preprocess text: tokenize, remove stopwords, and apply lemmatization.
    """
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    
    # Tokenize into sentences
    sentences = sent_tokenize(text.lower())
    
    # Tokenize and clean each sentence
    processed_sentences = []
    for sentence in sentences:
        # Tokenize into words
        words = word_tokenize(sentence)
        
        # Remove stopwords and non-alphabetic tokens
        words = [lemmatizer.lemmatize(word) for word in words 
                 if word.isalpha() and word not in stop_words and len(word) > 2]
        
        if len(words) > 0:  # Only add non-empty sentences
            processed_sentences.append(words)
    
    return processed_sentences

# Preprocess the text data
if text_data:
    sentences = preprocess_text(text_data)
    print(f"Total sentences after processing: {len(sentences)}")
    print(f"\nFirst 5 processed sentences:")
    for i, sentence in enumerate(sentences[:5]):
        print(f"{i+1}. {sentence}")

## 4. Train Word2Vec Model

Build and train a Word2Vec model using gensim with custom parameters like vector size, window size, minimum word count, and training epochs.

In [ ]:
# Train Word2Vec model
def train_word2vec_model(sentences, vector_size=100, window=5, min_count=3, epochs=10):
    """
    Train a Word2Vec model with custom parameters.
    
    Parameters:
    - vector_size: Dimension of word vectors (default: 100)
    - window: Context window size (default: 5)
    - min_count: Minimum word frequency (default: 3)
    - epochs: Number of training epochs (default: 10)
    """
    model = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=4,
        epochs=epochs,
        sg=0  # 0 for CBOW, 1 for Skip-gram
    )
    return model

# Train the model
if text_data and 'sentences' in locals():
    print("Training Word2Vec model...")
    w2v_model = train_word2vec_model(
        sentences,
        vector_size=100,
        window=5,
        min_count=2,
        epochs=10
    )
    print(f"Model trained successfully!")
    print(f"Vocabulary size: {len(w2v_model.wv)}")

## 5. Explore Word Embeddings

Analyze trained embeddings by finding similar words, computing word analogies, and visualizing embeddings using t-SNE or PCA.

In [ ]:
# Explore word similarities
def find_similar_words(model, word, topn=5):
    """
    Find similar words to a given word using cosine similarity.
    """
    if word not in model.wv:
        print(f"Word '{word}' not found in vocabulary.")
        return []
    
    similar_words = model.wv.most_similar(word, topn=topn)
    return similar_words

# Test similarity search
if 'w2v_model' in locals():
    print("Finding similar words...\n")
    
    # Get some words from vocabulary
    words_to_test = list(w2v_model.wv.index_to_key[:10])
    
    for word in words_to_test[:3]:
        similar = find_similar_words(w2v_model, word, topn=3)
        if similar:
            print(f"Words similar to '{word}':")
            for sim_word, score in similar:
                print(f"  - {sim_word}: {score:.4f}")
            print()

## 6. Visualize Word Embeddings

Visualize the word embeddings in 2D space using PCA to understand the relationships between words.

In [ ]:
# Visualize embeddings using PCA
def visualize_embeddings(model, num_words=20):
    """
    Visualize word embeddings in 2D space using PCA.
    """
    # Get the word vectors
    words = model.wv.index_to_key[:num_words]
    vectors = np.array([model.wv[word] for word in words])
    
    # Apply PCA to reduce to 2D
    pca = PCA(n_components=2)
    vectors_2d = pca.fit_transform(vectors)
    
    # Plot
    plt.figure(figsize=(10, 8))
    plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], alpha=0.7, s=100)
    
    for i, word in enumerate(words):
        plt.annotate(word, (vectors_2d[i, 0], vectors_2d[i, 1]),
                    fontsize=10, ha='center', va='center')
    
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
    plt.title('Word2Vec Embeddings Visualization (PCA)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Visualize the embeddings
if 'w2v_model' in locals():
    visualize_embeddings(w2v_model, num_words=20)

## 7. Save and Load Model

Persist the trained Word2Vec model to disk and demonstrate how to load and use it for downstream NLP tasks.

In [ ]:
# Save the model
def save_model(model, filepath):
    """
    Save the Word2Vec model to disk.
    """
    try:
        model.save(filepath)
        print(f"Model saved successfully to {filepath}")
    except Exception as e:
        print(f"Error saving model: {e}")

# Load the model
def load_model(filepath):
    """
    Load a Word2Vec model from disk.
    """
    try:
        model = Word2Vec.load(filepath)
        print(f"Model loaded successfully from {filepath}")
        return model
    except Exception as e:
        print(f"Error loading model: {e}")
        return None

# Save the model
if 'w2v_model' in locals():
    model_path = 'assignments/0607word2vec/word2vec_model.model'
    save_model(w2v_model, model_path)
    
    # Demonstrate loading the model
    loaded_model = load_model(model_path)
    if loaded_model:
        print(f"\nLoaded model vocabulary size: {len(loaded_model.wv)}")